# 18 — PORTABLE Final Inference for Grading

This notebook is intentionally **path-portable**.

It does **not** contain machine-specific paths such as `D:\medicine` or `C:\Users\...`.

Required inputs:
- `model.pkl` — the self-contained locked ensemble artifact produced before submission.
- organizer-provided unlabeled Test images.

The notebook searches for these inputs automatically in:
- current working directory and subdirectories
- notebook/repository directory and subdirectories
- `/kaggle/input` when running on Kaggle

Locked inference remains:
- DenseNet121
- 224×224
- grayscale → RGB
- seed-2026 70% + seed-3407 30%
- threshold 0.576
- TTA OFF

Output:
- exactly `Image_id,Prediction`

No training, no threshold tuning, and no test-label access.


In [ ]:
print('>>> CELL 1 START: portable environment', flush=True)
import os,sys,pickle,json,time,gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image,ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms,models

TEAM_ID='team01'  # change only if organizers require another exact filename

VALID_EXTENSIONS={'.png','.jpg','.jpeg','.bmp','.tif','.tiff'}
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
BATCH_SIZE=16

CWD=Path.cwd().resolve()
SEARCH_ROOTS=[CWD]

# Local repo / notebook parent candidates.
for p in [CWD.parent, CWD.parent.parent]:
    if p.exists() and p not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(p)

# Kaggle mounted input datasets.
KAGGLE_INPUT=Path('/kaggle/input')
if KAGGLE_INPUT.exists():
    SEARCH_ROOTS.append(KAGGLE_INPUT)

OUTPUT_DIR=Path('/kaggle/working') if Path('/kaggle/working').exists() else CWD/'final_submission_portable'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

print('Python:',sys.version.split()[0],flush=True)
print('PyTorch:',torch.__version__,flush=True)
print('Device:',DEVICE,flush=True)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('Search roots:',[str(x) for x in SEARCH_ROOTS],flush=True)
print('Output dir:',OUTPUT_DIR,flush=True)


In [ ]:
print('>>> CELL 2 START: auto-find model.pkl', flush=True)

def find_files_named(filename,roots,max_depth_hint=None):
    hits=[]
    seen=set()
    for root in roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob(filename):
                try:
                    rp=p.resolve()
                except Exception:
                    rp=p
                if p.is_file() and str(rp) not in seen:
                    seen.add(str(rp));hits.append(p)
        except PermissionError:
            pass
    return hits

model_hits=find_files_named('model.pkl',SEARCH_ROOTS)

if len(model_hits)==0:
    raise FileNotFoundError(
        'model.pkl not found. Put the submitted model.pkl in the notebook working directory '
        'or attach it as a Kaggle input dataset.'
    )

# Prefer a model.pkl located closest to current working directory.
model_hits=sorted(model_hits,key=lambda p:(len(p.parts),str(p).lower()))
MODEL_PATH=model_hits[0]

with open(MODEL_PATH,'rb') as f:
    restored=pickle.load(f)

assert restored['format']=='medical_challenge_2026_locked_ensemble_v1'
assert restored['architecture']=='DenseNet121'
assert restored['class_names']==['Normal','Pneumonia']
assert restored['image_size']==224
assert restored['tta'] is False
assert abs(float(restored['threshold'])-0.576)<1e-12
assert len(restored['seed_groups'])==2
assert sum(len(g['fold_state_dicts']) for g in restored['seed_groups'])==10
assert abs(sum(float(g['ensemble_weight']) for g in restored['seed_groups'])-1.0)<1e-9

print('MODEL_PATH:',MODEL_PATH,flush=True)
print('Model groups:',[(g['name'],g['ensemble_weight'],len(g['fold_state_dicts'])) for g in restored['seed_groups']],flush=True)
print('Threshold:',restored['threshold'],flush=True)


In [ ]:
print('>>> CELL 3 START: auto-find official unlabeled test directory', flush=True)

def image_paths(root):
    root=Path(root)
    return sorted(
        [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS],
        key=lambda p:str(p).lower()
    )

def candidate_test_dirs(roots):
    candidates=[]
    seen=set()
    preferred_names={'test','testing','official_test','official-test'}
    for root in roots:
        if not root.exists():
            continue
        # root itself may be test
        if root.name.lower() in preferred_names:
            imgs=image_paths(root)
            if imgs:
                rp=str(root.resolve())
                if rp not in seen:
                    seen.add(rp);candidates.append((root,len(imgs)))

        try:
            for p in root.rglob('*'):
                if not p.is_dir() or p.name.lower() not in preferred_names:
                    continue
                imgs=image_paths(p)
                if imgs:
                    rp=str(p.resolve())
                    if rp not in seen:
                        seen.add(rp);candidates.append((p,len(imgs)))
        except PermissionError:
            pass
    return candidates

candidates=candidate_test_dirs(SEARCH_ROOTS)

# Exclude obvious training/validation paths and any directory inside our local external sanity-check folder.
filtered=[]
for p,n in candidates:
    s=str(p).lower()
    if any(token in s for token in ['external_test_outputs','holdout_outputs','cv_outputs','train/','train\\','validation','/val/','\\val\\']):
        continue
    filtered.append((p,n))

if len(filtered)==0:
    raise FileNotFoundError(
        'Could not auto-detect an official test directory. '
        'Name the organizer test folder "test" or "official_test", or place it under the attached Kaggle dataset.'
    )

# If exactly one, use it. If several, display them and stop rather than silently selecting wrong data.
if len(filtered)>1:
    print('Multiple possible test directories found:',flush=True)
    for p,n in filtered:
        print(' ',p,'| images:',n,flush=True)
    raise RuntimeError(
        'Ambiguous test directory. Keep only the official organizer Test folder attached to this grading run.'
    )

TEST_DIR,n_test=filtered[0]
test_paths=image_paths(TEST_DIR)

# Guard against labeled folder structure: final official test should not expose class labels.
child_dir_names={p.name.lower() for p in TEST_DIR.iterdir() if p.is_dir()}
if 'normal' in child_dir_names or 'pneumonia' in child_dir_names:
    raise RuntimeError(
        'Detected NORMAL/PNEUMONIA subfolders inside TEST_DIR. '
        'This appears to be a labeled evaluation set, not the official unlabeled competition Test set.'
    )

ids=[p.name for p in test_paths]
if len(ids)!=len(set(ids)):
    raise RuntimeError('Duplicate base filenames detected; Image_id would be ambiguous.')

print('TEST_DIR:',TEST_DIR,flush=True)
print('Test images:',len(test_paths),flush=True)
print('First Image_id values:',ids[:5],flush=True)


In [ ]:
print('>>> CELL 4 START: deterministic preprocessing', flush=True)

IMAGE_SIZE=int(restored['image_size'])
THRESHOLD=float(restored['threshold'])
MEAN=restored['mean']
STD=restored['std']

class PadToSquare:
    def __call__(self,img):
        w,h=img.size
        side=max(w,h)
        left=(side-w)//2;right=side-w-left
        top=(side-h)//2;bottom=side-h-top
        return ImageOps.expand(img,border=(left,top,right,bottom),fill=0)

eval_tf=transforms.Compose([
    PadToSquare(),
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE),interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])

class TestDataset(Dataset):
    def __init__(self,paths): self.paths=list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        p=self.paths[i]
        with Image.open(p) as im:
            x=eval_tf(im.convert('L').convert('RGB'))
        return x,p.name

loader=DataLoader(TestDataset(test_paths),batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=False)

def make_model():
    m=models.densenet121(weights=None)
    m.classifier=nn.Linear(m.classifier.in_features,2)
    return m.to(DEVICE)

@torch.no_grad()
def infer_state(state):
    m=make_model()
    m.load_state_dict(state,strict=True)
    m.eval()
    probs=[];names=[]
    for x,n in loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            logits=m(x)
        probs.extend(torch.softmax(logits.float(),dim=1)[:,1].cpu().numpy())
        names.extend(list(n))
    del m
    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()
    return np.asarray(probs,float),names


In [ ]:
print('>>> CELL 5 START: locked ensemble inference', flush=True)

names_ref=None
group_probs={}

for group in restored['seed_groups']:
    fp=[]
    print('\nGroup:',group['name'],flush=True)
    for fold,state in enumerate(group['fold_state_dicts']):
        t0=time.time()
        p,names=infer_state(state)
        if names_ref is None:
            names_ref=names
        else:
            assert names_ref==names
        fp.append(p)
        print(f'  fold {fold}: {time.time()-t0:.1f}s',flush=True)
    group_probs[group['name']]=np.mean(np.stack(fp,axis=1),axis=1)

final_prob=np.zeros(len(test_paths),dtype=float)
for group in restored['seed_groups']:
    final_prob += float(group['ensemble_weight'])*group_probs[group['name']]

assert len(final_prob)==len(test_paths)
assert np.isfinite(final_prob).all()
assert names_ref==[p.name for p in test_paths]

idx=(final_prob>=THRESHOLD).astype(int)
labels=[restored['class_names'][i] for i in idx]

submission=pd.DataFrame({'Image_id':names_ref,'Prediction':labels})
assert submission.columns.tolist()==['Image_id','Prediction']
assert submission.Image_id.is_unique
assert set(submission.Prediction).issubset({'Normal','Pneumonia'})
assert len(submission)==len(test_paths)

CSV_PATH=OUTPUT_DIR/f'{TEAM_ID}.csv'
submission.to_csv(CSV_PATH,index=False)

roundtrip=pd.read_csv(CSV_PATH)
pd.testing.assert_frame_equal(submission,roundtrip)

print('\nCSV READY:',CSV_PATH,flush=True)
print('Rows:',len(submission),flush=True)
print('Counts:',submission.Prediction.value_counts().to_dict(),flush=True)
display(submission.head(10))


In [ ]:
print('>>> CELL 6 START: grading-safe checks', flush=True)

# Confirm no absolute Windows path is needed for inference.
summary={
    'status':'PORTABLE_FINAL_INFERENCE_PASS',
    'model_found_at_runtime':str(MODEL_PATH),
    'test_found_at_runtime':str(TEST_DIR),
    'n_test':len(test_paths),
    'architecture':restored['architecture'],
    'image_size':IMAGE_SIZE,
    'threshold':THRESHOLD,
    'tta':False,
    'model_groups':[
        {'name':g['name'],'weight':float(g['ensemble_weight']),'fold_models':len(g['fold_state_dicts'])}
        for g in restored['seed_groups']
    ],
    'csv':CSV_PATH.name,
    'columns':submission.columns.tolist(),
    'prediction_values':sorted(submission.Prediction.unique().tolist()),
}

with open(OUTPUT_DIR/'portable_run_summary.json','w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2)

print(json.dumps(summary,indent=2),flush=True)
print('\nPASS: notebook inference does not require D:\\medicine or C:\\Users\\Admin paths.',flush=True)


## Submission note

For the final package, prefer this notebook over notebook 16 for grader reruns.

Package:
- `model.pkl`
- `<TEAM_ID>.csv`
- `18_portable_final_inference.ipynb`

The notebook resolves paths at runtime rather than assuming the developer's Windows directory structure.
